# Stage 08 — Reporting

**Owner:** Pujan Dey (Visualisation & Reporting Lead)

**Objective.** Consolidate every artefact the live presentation needs:
- A single metrics summary table (RQ1 Simple LR + Multiple LR, RQ2 Naive Bayes).
- The paired-CV t-test results (RQ1 Simple vs Multiple LR, RQ2 NB vs majority baseline).
- Listing of every figure produced in `outputs/figures/`.
- A short narrative template linking findings back to Assessment 1 stakeholders.

**Inputs.** Checkpoints from notebooks 05, 06, 07.

**Outputs.**
- `outputs/tables/metrics_summary.csv`
- `outputs/tables/paired_ttest_results.json` (also written by notebook 07)

## 1. Setup and load all checkpoints

In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import config, reporting

reg_metrics   = json.load(open(config.CHECKPOINT_DIR / "05_reg_metrics.json"))
cls_metrics   = json.load(open(config.CHECKPOINT_DIR / "06_cls_metrics.json"))
ttest_results = json.load(open(config.CHECKPOINT_DIR / "07_ttest_results.json"))

simple_lr_metrics   = {k: v for k, v in reg_metrics["simple_lr"].items() if k != "predictor"}
multiple_lr_metrics = {k: v for k, v in reg_metrics["multiple_lr"].items() if k != "n_features"}
nb_metrics          = cls_metrics["naive_bayes"]
print("Loaded regression metrics, classification metrics, and t-test results.")

Loaded regression metrics, classification metrics, and t-test results.


## 2. Consolidated metrics summary

In [2]:
path = reporting.write_metrics_summary(simple_lr_metrics, multiple_lr_metrics, nb_metrics)
print("Written →", path)
pd.read_csv(path)

21:01:28 | INFO    | src.reporting | Metrics summary → D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\outputs\tables\metrics_summary.csv


Written → D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\outputs\tables\metrics_summary.csv


,model,MAE,RMSE,R2,Adj_R2,Accuracy,Precision,Recall,F1,ROC_AUC
0,Simple LR (RQ1 baseline),0.178168,0.218840,0.232429,0.231947,NaN,NaN,NaN,NaN,NaN
1,Multiple LR (RQ1 full),0.141916,0.182745,0.464746,0.456564,NaN,NaN,NaN,NaN,NaN
2,Naive Bayes (RQ2),NaN,NaN,NaN,NaN,0.752978,0.692754,0.904161,0.784464,0.857513


## 3. Paired t-test table (RQ1 + RQ2)

In [3]:
rows = []
for key, label in [
    ("rq1_simple_vs_multiple_lr",   "RQ1 — Simple LR vs Multiple LR (MAE)"),
    ("rq2_naive_bayes_vs_baseline", "RQ2 — Naive Bayes vs Majority baseline (accuracy)"),
]:
    r = ttest_results[key]
    rows.append({
        "comparison": label,
        "mean_a (model_a)": f"{r['mean_a']:.4f} ({r['model_a']})",
        "mean_b (model_b)": f"{r['mean_b']:.4f} ({r['model_b']})",
        "t_statistic": round(r["t_statistic"], 3),
        "p_value": round(r["p_value"], 4),
        "significant_at_0.05": r["significant_at_0.05"],
        "better_model": r["better_model"],
    })
pd.DataFrame(rows)

,comparison,mean_a (model_a),mean_b (model_b),t_statistic,p_value,significant_at_0.05,better_model
0,RQ1 — Simple LR vs Multiple LR (MAE),0.1784 (Simple LR),0.1432 (Multiple LR),18.571,0.0,True,Multiple LR
1,RQ2 — Naive Bayes vs Majority baseline (accuracy),0.7322 (NaiveBayes),0.5031 (MajorityBaseline),78.991,0.0,True,NaiveBayes


## 4. Figure inventory for the slides

In [4]:
figs = sorted(config.FIG_DIR.glob("*.png"))
pd.DataFrame({"figure": [f.name for f in figs],
              "size_kb": [round(f.stat().st_size / 1024, 1) for f in figs]})

,figure,size_kb
0,correlation_heatmap.png,386.7
1,rq1_coefficients.png,123.6
2,rq1_pred_vs_actual.png,193.2
3,rq1_qq_plot.png,56.2
4,rq1_residuals_vs_fitted.png,184.0
5,rq1_ttest_folds.png,43.9
6,rq2_confusion_matrix.png,24.8
7,rq2_ttest_folds.png,43.3
8,rq3_women_share_by_division.png,162.6
9,rq3_women_share_by_size.png,61.2


## Summary

All artefacts needed for the live presentation are now in `outputs/figures/` and `outputs/tables/`. The slide deck can be built directly from these files.